In [8]:
import torch
from PIL import Image
import requests
from transformers import AutoProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from huggingface_hub import login
from dotenv import load_dotenv
from openai import OpenAI
import logging
import os
from tqdm import tqdm
import json
import fitz

In [9]:
load_dotenv()
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

# Configure the huggingface_hub logger to suppress warnings
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# Login using the token
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

logging.getLogger("huggingface_hub").setLevel(logging.INFO)

## Using LLaVa-Next Model
https://huggingface.co/docs/transformers/model_doc/llava_next

In [ ]:
# Load model and processor
model_id = "llava-hf/llama3-llava-next-8b-hf"  # You can also try the larger variant
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id, 
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model-00001-of-00004.safetensors:   6%|6         | 315M/4.96G [00:00<?, ?B/s]

In [ ]:
def process_image_with_llava(image_path, prompt):
    # Load image
    if image_path.startswith('http'):
        image = Image.open(requests.get(image_path, stream=True).raw)
    else:
        image = Image.open(image_path)
    
    # Process inputs
    inputs = processor(prompt, image, return_tensors="pt").to(model.device)
    
    # Generate response
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )
    
    # Decode and return response
    response = processor.decode(output[0], skip_special_tokens=True)
    return response.strip()

## Using GPT-4o for Comparison

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def process_with_gpt(image_path, model="GPT4-mini"):
    # openai.api_key = api_key
    base64_image = encode_image(image_path)
    
    prompt = """
    This image contains Japanese text from a textbook with furigana (small kana above kanji).
    
    Task:
    1. Analyze the image and extract the main Japanese text
    2. IGNORE all furigana (small kana characters above kanji)
    3. Preserve correct sentence structure and punctuation
    4. Output the text in JSON format with this structure:
       {
         "chapter": "chapter_number",
         "exercises": [
           {
             "number": "exercise_number",
             "sentences": ["sentence1", "sentence2", ...]
           }
         ]
       }
    
    If chapter or exercise numbers aren't visible, use null values.
    Return ONLY the JSON output, nothing else.
    """
    
    response = openai.ChatCompletion.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]
            }
        ]
    )
    
    return response.choices[0].message.content

## Create a Specific Prompt for Japanese Text Extraction
Let's create a specialized prompt that asks the model to:

1. Ignore furigana
2. Preserve structure
3. Output in JSON format

In [ ]:
def extract_japanese_text(image_path):
    prompt = """
    This image contains Japanese text from a textbook with furigana (small kana above kanji).
    
    Task:
    1. Analyze the image and extract the main Japanese text
    2. IGNORE all furigana (small kana characters above kanji)
    3. Preserve correct sentence structure and punctuation
    4. Output the text in JSON format with this structure:
       {
         "chapter": "chapter_number",
         "exercises": [
           {
             "number": "exercise_number",
             "sentences": ["sentence1", "sentence2", ...]
           }
         ]
       }
    
    If chapter or exercise numbers aren't visible, use null values.
    Return ONLY the JSON output, nothing else.
    """
    
    result = process_image_with_llava(image_path, prompt)
    return result

## Process PDF Pages

In [ ]:
def process_pdf(pdf_path, output_dir):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Open PDF
    doc = fitz.open(pdf_path)
    all_results = []
    
    # Process each page
    for page_num in tqdm(range(len(doc)), desc="Processing pages"):
        page = doc[page_num]
        
        # Convert page to image
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for better quality
        img_path = f"{output_dir}/page_{page_num}.png"
        pix.save(img_path)
        
        # Process with LLaVA-NeXT
        result = extract_japanese_text(img_path)
        
        # Try to parse the result as JSON
        try:
            json_result = json.loads(result)
            json_result["page_number"] = page_num + 1
            all_results.append(json_result)
        except json.JSONDecodeError:
            print(f"Could not parse JSON from page {page_num+1}. Raw output: {result[:100]}...")
    
    # Save all results
    with open(f"{output_dir}/extracted_text.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    
    return all_results

## Validation and Post-Processing
Add a validation step to ensure the JSON output is correctly formatted:

In [ ]:
def validate_and_clean_results(all_results):
    cleaned_results = []
    
    for page_result in all_results:
        if isinstance(page_result, dict) and "exercises" in page_result:
            cleaned_results.append(page_result)
        else:
            print(f"Skipping invalid result for page {page_result.get('page_number', 'unknown')}")
    
    # Merge results by chapter
    merged_results = {}
    for result in cleaned_results:
        chapter = result.get("chapter")
        if chapter not in merged_results:
            merged_results[chapter] = {"chapter": chapter, "exercises": []}
        
        # Append exercises
        merged_results[chapter]["exercises"].extend(result.get("exercises", []))
    
    return list(merged_results.values())

# Full Pipeline
Putting it all together

In [ ]:
def process_file(pdf_path, output_dir="output"):
    print(f"Processing PDF: {pdf_path}")
    
    # Process PDF
    raw_results = process_pdf(pdf_path, output_dir)
    
    # Validate and clean results
    final_results = validate_and_clean_results(raw_results)
    
    # Save final results
    with open(f"{output_dir}/final_structured_text.json", "w", encoding="utf-8") as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    
    print(f"Processing complete. Results saved to {output_dir}/final_structured_text.json")
    return final_results

In [ ]:
filePath = ""
process_file(filePath)